# Per-example gradient clipping with vmap

Differentially private SGD clips **each example's** gradient before averaging, so no single example can move the model by more than a bound $C$. Frameworks that only give the batch gradient make this awkward. In JAX it is `vmap` over `grad`.

Model and loss for one example $(x, y)$, with `params = {'w': (D,), 'b': ()}`:

$$\ell(w, b; x, y) = \tfrac12\,(x^\top w + b - y)^2$$

Write `clipped_mean_grad(params, x, y, max_norm)`:

1. Compute the per-example gradients $g_i = \nabla_{(w,b)}\, \ell(\cdot\,; x_i, y_i)$ for every row of `x` `(B, D)` and `y` `(B,)`.
2. Each example's norm is the global L2 norm over both leaves: $n_i = \sqrt{\lVert g_{i,w}\rVert^2 + g_{i,b}^2}$.
3. Scale each gradient by $\min(1, C / n_i)$, using factor `1` when $n_i = 0$.
4. Average the clipped gradients over the batch.

Return `(grads, norms)`: `grads` is a dict `{'w': (D,), 'b': ()}` and `norms` is `(B,)` (the norms before clipping). float32. No Python loop over examples.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def clipped_mean_grad(params, x, y, max_norm):
    """params {'w': (D,), 'b': ()}. Return (mean clipped grads dict, per-example norms (B,))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["w", {"__nd__": [0.0, 0.0], "dtype": "float64"}], ["b", {"__nd__": 0.0, "dtype": "float64"}]]}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [1.0, 2.0], "dtype": "float64"}, 10.0], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": -1.5, "dtype": "float32"}], ["w", {"__nd__": [-0.5, -1.0], "dtype": "float32"}]]}, {"__nd__": [1.4142135381698608, 2.8284270763397217], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [0.0, 0.0], "dtype": "float64"}], ["b", {"__nd__": 0.0, "dtype": "float64"}]]}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [1.0, 2.0], "dtype": "float64"}, 1.0], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": -0.7071067690849304, "dtype": "float32"}], ["w", {"__nd__": [-0.3535533845424652, -0.3535533845424652], "dtype": "float32"}]]}, {"__nd__": [1.4142135381698608, 2.8284270763397217], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [1.0, -1.0], "dtype": "float64"}], ["b", {"__nd__": 0.5, "dtype": "float64"}]]}, {"__nd__": [[1.0, 1.0], [2.0, 1.0]], "dtype": "float64"}, {"__nd__": [0.5, 1.5], "dtype": "float64"}, 0.1], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": 0.0, "dtype": "float32"}], ["w", {"__nd__": [0.0, 0.0], "dtype": "float32"}]]}, {"__nd__": [0.0, 0.0], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [0.35877340800391416, 1.5106773081434572, -1.7863312373111822, 1.686613508665752], "dtype": "float64"}], ["b", {"__nd__": 0.0, "dtype": "float64"}]]}, {"__nd__": [[-0.047317212156137566, -0.7999785843751532, -0.802956718390362, -1.082816516582591], [-0.22364535840745078, 0.8338841795521574, 0.5840637316429703, 0.6382860242561829], [-1.6948083917690129, -1.5709621176914044, 1.553803174289652, 0.96888536545666], [2.18321400147264, 1.2098158348986146, -1.0243913350545566, 1.2852724548122736], [0.6283942642703312, 0.21482492767891934, -0.8198733225718833, 0.0027802218072239555], [-0.14699870178042784, 0.8925351116148221, 0.12449694399871385, 1.5847459143396285], [-0.7744857532181485, 0.4711025594104292, 0.49884237745245574, -0.864338539814181], [-1.1812660464710163, 1.013579500325613, 0.5174404713814393, -0.1625051850123894]], "dtype": "float64"}, {"__nd__": [0.3943673986045515, -0.6753440287843157, 0.2383482020485698, -0.6051266946615304, 0.18878846665371848, 0.9860894628834065, -0.33963024501217354, 0.8497738099419226], "dtype": "float64"}, 1.0], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": 0.018113911151885986, "dtype": "float32"}], ["w", {"__nd__": [0.31960123777389526, 0.20202314853668213, -0.13655152916908264, 0.3055459260940552], "dtype": "float32"}]]}, {"__nd__": [3.7418580055236816, 2.98163104057312, 13.577872276306152, 22.73334312438965, 2.6609137058258057, 5.753150939941406, 2.6445529460906982, 1.8137754201889038], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [-1.0341993407801506, -0.2884907696501737, -1.3647425964001083, -0.10058608537896181], "dtype": "float64"}], ["b", {"__nd__": 1.0, "dtype": "float64"}]]}, {"__nd__": [[0.39327041662615714, -0.9863133144860667, -1.3383785835371074, 6.70097345051987], [7.118964184212039, -6.078155024732625, 5.870572466279473, -8.61534806435496], [-3.672396254185228, -0.8672279796087845, 0.555303013356508, -5.237083380021228], [-3.612343897033792, 0.38326909371718443, -3.467113616913577, -3.206003946301494], [3.3313172611198554, 4.658605264179598, -0.3002751554004135, 0.09098956296121667], [-0.09799058582370858, -2.272441466375869, -2.5569660704577153, -5.188722949604353], [3.2419436099711985, -0.42079835155780865, -0.6871969904834997, 1.0027130660791437], [2.653374212734687, 7.105172511893754, -1.3677290979465013, -2.0058613314020866]], "dtype": "float64"}, {"__nd__": [0.20057350347744685, -0.8455026583233285, 1.2724301176555226, 0.9262440047595556, 0.6939247628614134, -0.1196149568737294, 0.739582174610242, 1.0502047166386168], "dtype": "float64"}, 0.5], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": -0.001444422174245119, "dtype": "float32"}], ["w", {"__nd__": [-0.2137821465730667, -0.10672982037067413, -0.06834105402231216, -0.038967035710811615], "dtype": "float32"}]]}, {"__nd__": [12.785049438476562, 153.23146057128906, 23.236982345581055, 52.879638671875, 23.769319534301758, 37.06877136230469, 7.742954254150391, 22.278322219848633], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [0.4999059994351346, -0.2799827475865304, -1.6850729956882953], "dtype": "float64"}], ["b", {"__nd__": -0.2, "dtype": "float64"}]]}, {"__nd__": [[0.9469623538219463, -0.3532509656510693, -0.8432343818323768], [0.8905983820560427, 0.33817143735531163, -1.4188364513624918], [-0.09904137373005252, -0.2692340321763984, 0.9926118054712234], [-1.895121829358109, -0.3855607500329723, -0.1382491179542964], [1.4189004130302252, -1.0833898591198583, -2.940431935506099], [-1.3840283315854316, 0.263069762783062, 0.809592098994733], [1.7335425693272173, 0.40093512304012024, -0.8023103035286667], [0.8843895901120883, -0.011057745469750928, -0.2009579446051938], [0.017552546089036017, 1.2745760509101105, -0.9014413693567734], [-1.0239272781381537, -0.06652210021038106, -0.06656995155514676], [-0.8144586293391828, -0.8794565088883497, 0.37765916004244404], [1.337835134332041, 0.7129918729459087, 0.6851211109663922], [1.2457373766175497, -1.99097272670123, -1.5103268361371722], [-0.7053981939290472, -1.1951111440348927, -0.9893977243928588], [0.06987400383657381, -1.2843896578479024, -0.44349701839532474], [-2.0431546819494004, -0.9739745699555712, 1.345547885926575]], "dtype": "float64"}, {"__nd__": [-0.5428950996559225, 0.3241388671590421, 0.7081542141604098, -0.49090235523671877, 0.579084730531848, 0.21453528542881106, 0.6974051956994979, -0.6355527655967318, -1.4401713176798854, -1.0221198928506263, -0.554816582754259, 0.5358286946534183, -0.20430898694963992, -0.39880631402193534, -0.1796813723272528, 1.0997568665514252], "dtype": "float64"}, 1000000.0], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": 0.6329385042190552, "dtype": "float32"}], ["w", {"__nd__": [1.8364827632904053, -0.5960220694541931, -2.7030248641967773], "dtype": "float32"}]]}, {"__nd__": [3.8616907596588135, 4.390254497528076, 3.6737186908721924, 0.6884390711784363, 18.586624145507812, 4.85432767868042, 2.651219367980957, 1.6462438106536865, 4.470231056213379, 0.6326122879981995, 0.7106581330299377, 2.7584612369537354, 11.061577796936035, 3.651728630065918, 1.893791675567627, 12.156146049499512], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w", {"__nd__": [-0.5754988975296205, 1.511331668915082, -1.227592845758537, -0.5922351807140467, 0.855002651076008], "dtype": "float64"}], ["b", {"__nd__": 0.3, "dtype": "float64"}]]}, {"__nd__": [[0.5178268740164172, -0.8555092830790272, -0.10903398836430599, -0.7089297260980899, 0.8510212769423531], [0.5397406470269119, -0.7131092966031697, 0.3971833443260635, -0.3671460370746888, -0.07191657774926997], [0.06461484502610823, -0.5640442432917747, 0.4250072828482813, 0.46028239783009683, 0.42593165582019393], [1.1046397070960905, -0.27174114252366427, -1.4828593634501943, -0.489072005359709, 0.2007368892738041], [0.02198571377633105, 0.5786880417250706, 0.8064177974523544, -0.11758989750756722, -0.11037013953690167], [-0.9874075351661735, 1.0330955204913965, -0.26393657528783593, -1.0833076865189253, -1.114373881782793], [-0.4147657766285512, 1.5553992101837082, 0.7725462701194257, -1.6940422362384258, -0.12361444090580717], [0.2639596332433514, 0.6265573247505951, 0.3436628055732324, -0.6407415119480926, 0.4118864085570845], [-0.5541505601270842, -2.2406484235127633, -0.7088634775653162, -0.6859673154744348, 0.9536290088133706], [1.0560046751839265, -1.045279424641642, 0.17390955321529164, 0.09013529248460428, -1.435795498095001], [-2.0531366566250346, -0.09276540124128595, 0.13644372719186973, -0.5948320751120193, -2.7111664954113537], [-0.029483978631708624, -0.5786198747278738, -0.8894063342725544, -0.32418700968801495, -1.12855669323536], [0.5598427280686814, 0.010603864924009842, -0.5594566369571077, 0.2818726076342451, -0.032100222264474815], [-1.518206205158868, -0.1441467553139046, -0.4571565407824213, -1.7407156772498007, -0.6520424779536729], [0.42634016461290364, -0.6194956228417741, 1.4075074257264955, 1.6982164351447544, -0.3063453638612675], [0.553434424296176, 1.9926526478415145, -0.331526522350224, -0.8989570837450346, -0.422331995147081], [-1.3742214526554224, -1.3491851888620348, 0.2086518433004105, -0.4167563978840294, 0.24644175655585768], [0.4883741105141886, -0.7643160931773055, -0.1944523423331713, -0.2693043085863964, -0.6686649004393013], [-0.8364556720740618, 0.3155505374569986, -0.14024643621485902, 0.3322867996571848, 1.0009222444961468], [1.0142174316108183, -1.1120591826841941, -0.8558236870860543, -0.2562514323596529, 0.1236371145358894], [0.9383854586360836, -0.9621771389420167, -0.43685307558735226, 0.24703569469092412, 1.2288651191177333], [1.3792342883219184, -1.0059765041187922, -0.21203724299336404, -0.8077502873624303, 0.8474791280591054], [0.4469000649128899, -0.35779615802814096, -0.6112468227277504, 0.2949523400999831, 0.9802940600621635], [-0.767444144080219, -0.3236618111270047, 0.1129737681875106, 0.018011866501908936, 0.3378669966589955], [-0.9995766587698955, -0.5949276168950793, -2.1001051526512886, 0.05573366268881772, 1.0701569228810817], [-2.057195404042072, -0.6318079969806747, 0.8985241778631176, -0.1428027365344208, -1.8950428195675442], [1.168788331744318, -1.4259696378610343, 1.3709104657520208, -0.58464059879444, -0.7065454379067378], [0.026944605818733074, -0.022175546391032475, 0.6004755228520038, 0.5025060567088484, -0.14793754391747488], [1.3225328287068747, 0.7134748212984285, 0.27379161855350254, 0.13511514244719608, -0.09532521278129974], [-0.07758069467940808, 0.6648989933924994, 0.4742533005081589, 0.0397348380155517, 1.0151129233973155], [-0.26221152794886715, 0.43589613755299295, 0.8595237833519895, -1.1155043350355576, -0.2641148710183764], [1.6688064665782514, -0.9571035511645589, 1.1520796067501626, -1.0597628522710998, 0.405535390337305]], "dtype": "float64"}, {"__nd__": [-10.641892166186686, -6.163888649025725, -3.7531224638067915, -5.4882036330077835, 10.085690589142688, 8.334942966216094, 8.700599664307159, 1.9348830714441678, 16.366916490384465, 10.287923702896826, 3.247528649959841, -10.487857298833259, 6.8544661416136226, 6.521584987897013, 0.9434243617109992, 7.128873475063768, 7.699365242725696, -2.9845955358926544, 5.6529511106599175, 3.47065490112703, 3.131496577718769, -3.4179643371544444, -4.629779346452414, 1.1862428149034983, 14.317178577246315, 23.259854226861393, -3.698057478971567, -10.543470939889712, -3.4156052722203585, -8.351430939597623, -4.170599012110719, 3.9678218231934848], "dtype": "float64"}, 2.0], "e": {"__tup__": [{"__dict__": [["b", {"__nd__": -0.08985636383295059, "dtype": "float32"}], ["w", {"__nd__": [0.2440294325351715, 0.004311209544539452, -0.016613874584436417, 0.08148731291294098, 0.12557606399059296], "dtype": "float32"}]]}, {"__nd__": [19.134357452392578, 6.870570659637451, 3.7638819217681885, 15.343283653259277, 14.096054077148438, 13.853528022766113, 15.590178489685059, 0.7791845798492432, 48.83232116699219, 31.493120193481445, 14.53453254699707, 19.189504623413086, 8.320252418518066, 11.979560852050781, 12.313119888305664, 8.981179237365723, 18.86117935180664, 2.5839710235595703, 6.100945472717285, 8.334704399108887, 7.235082149505615, 6.619215488433838, 9.195347785949707, 1.0717408657073975, 30.606143951416016, 80.42643737792969, 2.0558388233184814, 12.32219123840332, 6.480480670928955, 16.386476516723633, 8.480167388916016, 17.620635986328125], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(clipped_mean_grad, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")